In [ ]:
dayobs = 20250827

## Imports

In [ ]:
import asyncio
import lsst_efd_client

import astropy.units as u
from astropy.coordinates import get_sun, AltAz, EarthLocation
from astropy.time import Time, TimeDelta
from datetime import datetime, timezone, timedelta
import galsim
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytz
import sys
import matplotlib.dates as mdates

%matplotlib inline

In [ ]:
from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
import os

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

## Define functions

In [ ]:
def getPsfGradPerZernike(
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
    jmax: int = 22,
) -> np.ndarray:
    """Get the gradient of the PSF FWHM with respect to each Zernike.

    This function takes no positional arguments. All parameters must be passed
    by name (see the list of parameters below).

    Parameters
    ----------
    diameter : float, optional
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float, optional
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int, optional
        The minimum Noll index, inclusive. Must be >= 0. (the default is 4)
    jmax : int, optional
        The max Zernike Noll index, inclusive. Must be >= jmin.
        (the default is 22.)

    Returns
    -------
    np.ndarray
        Gradient of the PSF FWHM with respect to the corresponding Zernike.
        Units are arcsec / micron.

    Raises
    ------
    ValueError
        If jmin is negative or jmax is less than jmin
    """
    # Check jmin and jmax
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")
    if jmax < jmin:
        raise ValueError("jmax must be greater than jmin.")

    # Calculate the conversion factors
    conversion_factors = np.zeros(jmax + 1)
    for i in range(jmin, jmax + 1):
        # Set coefficients for this Noll index: coefs = [0, 0, ..., 1]
        # Note the first coefficient is Noll index 0, which does not exist and
        # is therefore always ignored by galsim
        coefs = [0] * i + [1]

        # Create the Zernike polynomial with these coefficients
        R_outer = diameter / 2
        R_inner = R_outer * obscuration
        Z = galsim.zernike.Zernike(coefs, R_outer=R_outer, R_inner=R_inner)

        # We can calculate the size of the PSF from the RMS of the gradient of
        # the wavefront. The gradient of the wavefront perturbs photon paths.
        # The RMS quantifies the size of the collective perturbation.
        # If we expand the wavefront gradient in another series of Zernike
        # polynomials, we can exploit the orthonormality of the Zernikes to
        # calculate the RMS from the Zernike coefficients.
        rms_tilt = np.sqrt(np.sum(Z.gradX.coef**2 + Z.gradY.coef**2) / 2)

        # Convert to arcsec per micron
        rms_tilt = np.rad2deg(rms_tilt * 1e-6) * 3600

        # Convert rms -> fwhm
        fwhm_tilt = 2 * np.sqrt(2 * np.log(2)) * rms_tilt

        # Save this conversion factor
        conversion_factors[i] = fwhm_tilt

    return conversion_factors[jmin:]


def convertZernikesToPsfWidth(
    zernikes: np.ndarray,
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
) -> np.ndarray:
    """Convert Zernike amplitudes to quadrature contribution to the PSF FWHM.

    Parameters
    ----------
    zernikes : np.ndarray
        Zernike amplitudes (in microns), starting with Noll index `jmin`.
        Either a 1D array of zernike amplitudes, or a 2D array, where each row
        corresponds to a different set of amplitudes.
    diameter : float
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int
        The minimum Zernike Noll index, inclusive. Must be >= 0. The
        max Noll index is inferred from `jmin` and the length of `zernikes`.
        (the default is 4, which ignores piston, x & y offsets, and tilt.)

    Returns
    -------
    dFWHM: np.ndarray
        Quadrature contribution of each Zernike vector to the PSF FWHM
        (in arcseconds).

    Notes
    -----
    Converting Zernike amplitudes to their quadrature contributions to the PSF
    FWHM allows for easier physical interpretation of Zernike amplitudes and
    the performance of the AOS system.

    For example, image we have a true set of zernikes, [Z4, Z5, Z6], such that
    ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.2, 0.3] arcsecs.
    These Zernike perturbations increase the PSF FWHM by
    sqrt[(0.1)^2 + (-0.2)^2 + (0.3)^2] ~ 0.37 arcsecs.

    If the AOS perfectly corrects for these perturbations, the PSF FWHM will
    not increase in size. However, imagine the AOS estimates zernikes, such
    that ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.3, 0.4] arcsecs.
    These estimated Zernikes, do not exactly match the true Zernikes above.
    Therefore, the post-correction PSF will still be degraded with respect to
    the optimal PSF. In particular, the PSF FWHM will be increased by
    sqrt[(0.1 - 0.1)^2 + (-0.2 - (-0.3))^2 + (0.3 - 0.4)^2] ~ 0.14 arcsecs.

    This conversion depends on a linear approximation that begins to break down
    for RSS(dFWHM) > 0.20 arcsecs. Beyond this point, the approximation tends
    to overestimate the PSF degradation. In other words, if
    sqrt(sum( dFWHM^2 )) > 0.20 arcsec, it is likely that dFWHM is
    over-estimated. However, the point beyond which this breakdown begins
    (and whether the approximation over- or under-estimates dFWHM) can change,
    depending on which Zernikes have large amplitudes. In general, if you have
    large Zernike amplitudes, proceed with caution!
    Note that if the amplitudes Z_est and Z_true are large, this is okay, as
    long as |Z_est - Z_true| is small.

    For a notebook demonstrating where the approximation breaks down:
    https://gist.github.com/jfcrenshaw/24056516cfa3ce0237e39507674a43e1

    Raises
    ------
    ValueError
        If jmin is negative
    """
    # Check jmin
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")

    # Calculate jmax from jmin and the length of the zernike array
    jmax = jmin + np.array(zernikes).shape[-1] - 1

    # Calculate the conversion factors for each zernike
    conversion_factors = getPsfGradPerZernike(
        jmin=jmin,
        jmax=jmax,
        diameter=diameter,
        obscuration=obscuration,
    )

    # Convert the Zernike amplitudes from microns to their quadrature
    # contribution to the PSF FWHM
    dFWHM = conversion_factors * zernikes

    return dFWHM

In [ ]:
def convert_dataframe_to_timezone(
    df: pd.DataFrame, target_tz: str = "America/Santiago", timestamp_column: str = "timestamp"
) -> pd.DataFrame:
    """Convert a timestamp column or index to be timezone-aware.

    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame.
    target_tz : str
        The target timezone (e.g., "America/Santiago").
    timestamp_column : str
        Name of the timestamp column, if index is not already datetime-like.

    Returns
    -------
    pd.DataFrame
        A copy of the DataFrame with a localized DatetimeIndex.
    """
    if df.empty:
        return df.copy()

    df = df.copy()

    if isinstance(df.index, pd.DatetimeIndex):
        df.index = (
            df.index.tz_convert(target_tz)
            if df.index.tzinfo
            else df.index.tz_localize("UTC").tz_convert(target_tz)
        )
    elif timestamp_column in df.columns:
        df[timestamp_column] = pd.to_datetime(df[timestamp_column], utc=True, format='ISO8601')
        df.set_index(timestamp_column, inplace=True)
        df.index = df.index.tz_convert(target_tz)
    else:
        raise ValueError("DataFrame has no datetime index and no valid timestamp column.")

    return df


async def query_efd_grouped(client: lsst_efd_client.EfdClient, start_date: Time, end_date: Time) -> pd.DataFrame:
    topic_name = "lsst.sal.MTM1M3TS.thermalData"
    fields = [f"mean(absoluteTemperature{i})" for i in range(96)]
    query = client.build_time_range_query(
        topic_name,
        fields,
        start_date,
        end_date,
    )
    return await client._do_query(query + " GROUP BY time(1m)")


async def query_m1m3_ess(
    client: lsst_efd_client.EfdClient, start_date: Time, end_date: Time
) -> list[pd.DataFrame]:
    data_frames = []

    ess_start = 114
    ess_end = 117
    num_channels = (16, 16, 16, 16, 16, 15)
    for ess_index in range(ess_start, ess_end + 1):
        topic_name = "lsst.sal.ESS.temperature"
        for i in range(1, 7):
            fields = [f"mean(temperatureItem{i}) AS ch{i+1}" for i in range(num_channels[i - 1])]
            query = client.build_time_range_query(topic_name, fields, start_date, end_date, index=ess_index)
            query += f" AND salIndex = {ess_index}"
            query += f" AND sensorName =~ /{i}\\/6/ GROUP BY time(1m)"
            table = await client._do_query(query)
            table = table.add_prefix(f"ess{ess_index}_{i}_")
            data_frames.append(table)

    return pd.concat(data_frames, axis=1)

async def query_tma_truss_ess(
    client: lsst_efd_client.EfdClient, start_date: Time, end_date: Time
) -> list[pd.DataFrame]:
    data_frames = []

    topic_name = "lsst.sal.ESS.temperature"
    ess_index = 2
    name_dict = {6:'tma_truss_+x_+y', 7:'tma_truss_-x_-y'}
    for i in range(6, 8):
        fields = [f"mean(temperatureItem{i}) AS ch{i}"]
        query = client.build_time_range_query(topic_name, fields, start_date, end_date, index=ess_index)
        query += f" GROUP BY time(1m)"
        table = await client._do_query(query)
        table.rename(columns={f'ch{i}': name_dict[i]}, inplace=True)
        # table = table.add_prefix(f"ess{ess_index}_{i}_")
        data_frames.append(table)

    return pd.concat(data_frames, axis=1)

async def query_m2(client: lsst_efd_client.EfdClient, start_date: Time, end_date: Time) -> pd.DataFrame:
    """Query the EFD for MTM2 temperature data.

    This function obtains temperature data from the MTM2.temperature topic.

    Parameters
    ----------
    client : lsst_efd_client.EfdClient
        The EFD client to use for querying.
    start_date : Time
        The start date for the query.
    end_date : Time
        The end date for the query.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the mean temperatures for rings 0-11 (excluding ring5)
        and ring5 as a separate column, averaged over 1-minute intervals.
    """
    topic_name = "lsst.sal.MTM2.temperature"
    fields = [f"mean(ring{i})" for i in range(12)]
    query = client.build_time_range_query(
        topic_name,
        fields,
        start_date,
        end_date,
    )
    df = await client._do_query(query + " GROUP BY time(1m)")
    
    # Separate ring5 from the other rings
    ring5_data = df["mean_5"]
    other_rings = df.drop(columns=["mean_5"])
    
    # Calculate average of all rings except ring5
    temperature_avg = other_rings.mean(axis=1)
    
    # Create result DataFrame with both columns
    result = pd.DataFrame({
        "temperature": temperature_avg,
        "ring5": ring5_data
    })
    
    return result


async def query_efd(start_date: Time, end_date: Time) -> dict:
    # Define a time window for the previous day
    # client = lsst_efd_client.EfdClient("summit_efd")
    client = lsst_efd_client.EfdClient("usdf_efd", output_mode="dataframe")

    # Collect weather station temperatures
    topic = "lsst.sal.ESS.temperature"
    fields = ["temperatureItem0"]
    temperature_outdoor = await client.select_time_series(topic, fields, start_date, end_date, index=301)
    temperature_indoor = await client.select_time_series(topic, fields, start_date, end_date, index=112)

    # Collect M1M3TS temperatures
    temperature_mtm1m3ts = await query_m1m3_ess(client, start_date, end_date)

    # Collect MTM2 temperatures
    temperature_mtm2 = await query_m2(client, start_date, end_date)

    # Collect TMA truss temperatures
    temperature_tma_truss = await query_tma_truss_ess(client, start_date, end_date)

    # Collect top end temperatures
    topic = "lsst.sal.MTMount.topEndChiller"
    fields = ["ambientTemperatureSensor0502"]
    temperature_topend = await client.select_time_series(topic, fields, start_date, end_date)

    # Collect MTM1M3TS setpoints
    topic = "lsst.sal.MTM1M3TS.command_applySetpoints"
    fields = ["heatersSetpoint", "private_identity"]
    mtm1m3ts_setpoints = await client.select_time_series(topic, fields, start_date, end_date)

    # Collect top end setpoints
    topic = "lsst.sal.MTMount.command_setThermal"
    fields = ["topEndChillerSetpoint", "private_identity"]
    topend_setpoints = await client.select_time_series(topic, fields, start_date, end_date)

    # Collect HVAC setpoints
    topic = "lsst.sal.HVAC.command_configLowerAhu"
    fields = ["workingSetpoint", "private_identity"]
    hvac_setpoints = await client.select_time_series(topic, fields, start_date, end_date)

    # Is dome open?
    topic = "lsst.sal.MTDome.apertureShutter"
    fields = ["positionActual0", "positionActual1"]
    shutter = await client.select_time_series(topic, fields, start_date, end_date)

    temperature_outdoor = convert_dataframe_to_timezone(temperature_outdoor)
    temperature_indoor = convert_dataframe_to_timezone(temperature_indoor)
    temperature_mtm1m3ts = convert_dataframe_to_timezone(temperature_mtm1m3ts)
    temperature_mtm2 = convert_dataframe_to_timezone(temperature_mtm2)
    temperature_tma_truss = convert_dataframe_to_timezone(temperature_tma_truss)
    temerature_topend = convert_dataframe_to_timezone(temperature_topend)
    mtm1m3ts_setpoints = convert_dataframe_to_timezone(mtm1m3ts_setpoints)
    topend_setpoints = convert_dataframe_to_timezone(topend_setpoints)
    hvac_setpoints = convert_dataframe_to_timezone(hvac_setpoints)
    shutter = convert_dataframe_to_timezone(shutter)

    return {
        "temperature_outdoor": temperature_outdoor,
        "temperature_indoor": temperature_indoor,
        "temperature_mtm1m3ts": temperature_mtm1m3ts,
        "temperature_mtm2": temperature_mtm2,
        "temperature_tma_truss": temperature_tma_truss,
        "temperature_topend": temperature_topend,
        "mtm1m3ts_setpoints": mtm1m3ts_setpoints,
        "topend_setpoints": topend_setpoints,
        "hvac_setpoints": hvac_setpoints,
        "shutter": shutter,
    }


def get_shutter_closed_intervals(shutter: pd.DataFrame) -> list[tuple[pd.Timestamp | None, pd.Timestamp | None]]:
    """Find time intervals when the shutter was closed.

    Parameters
    ----------
    shutter : pd.DataFrame
        EFD archival data containing positionActual0 and positionActual1
        telemetry.

    Returns
    -------
    list[tuple[pd.Timestamp | None, pd.Timestamp | None]]
        The beginning and end of each interval in the dataset during which the
        dome was closed.
    """
    closed_intervals = []
    current_start = None

    shutter_is_open = (shutter["positionActual0"] >= 50) | (shutter["positionActual1"] >= 50)

    for time, is_open in shutter_is_open.items():
        if not is_open:
            if current_start is None:
                current_start = time  # mark the beginning of a closed period
        else:
            if current_start is not None:
                closed_intervals.append((current_start, time))
                current_start = None

    # Handle case where dome was closed until the end
    if current_start is not None:
        closed_intervals.append((current_start, shutter_is_open.index[-1]))

    return closed_intervals

def get_survey_intervals(cons_db_data: pd.DataFrame) -> list[tuple[pd.Timestamp | None, pd.Timestamp | None]]:
    """Find time intervals when survey observations were taken.

    Parameters
    ----------
    cons_db_data : pd.DataFrame
        ConsDB archival data containing positionActual0 and positionActual1
        telemetry.

    Returns
    -------
    list[tuple[pd.Timestamp | None, pd.Timestamp | None]]
        The beginning and end of each interval in the dataset during which the
        survey data was taken.
    """
    survey_intervals = []
    current_start = None

    # Include all possible FBS-driven blocks as "survey" data
    survey_block_names = ["BLOCK-365", "BLOCK-407", "BLOCK-408"]
    survey_data = cons_db_data["science_program"].isin(survey_block_names)

    for time, is_survey in survey_data.items():
        if is_survey is True:
            if current_start is None:
                current_start = time  # mark the beginning of an survey block
        else:
            if current_start is not None:
                survey_intervals.append((current_start, time))
                current_start = None

    # Handle case where survey was run until end of night
    if current_start is not None:
        survey_intervals.append((current_start, survey_data.index[-1]))

    return survey_intervals


def find_sun_altitude_crossings(start_time: Time, end_time: Time, altitude_deg: float, rising: bool) -> list:
    """
    Return a list of localized datetimes when the sun crosses a given altitude
    (e.g., -18 for twilight, 0 for sunrise/sunset) between start_time and end_time.

    Parameters
    ----------
    start_time : Time
        Astropy start time.
    end_time : Time
        Astropy end time.
    altitude_deg : float
        Target sun altitude in degrees.
    rising : bool
        True for rising (upward crossing), False for setting (downward).

    Returns
    -------
    list[datetime]
        Localized CLT datetimes when the sun crosses the given altitude.
    """
    location = EarthLocation.of_site("Cerro Pachon")
    clt = pytz.timezone("America/Santiago")

    step_seconds = 60
    n_steps = int((end_time - start_time).sec / step_seconds) + 1
    times = start_time + TimeDelta(np.arange(n_steps) * step_seconds, format='sec')

    altaz = AltAz(obstime=times, location=location)
    sun_alt = get_sun(times).transform_to(altaz).alt.deg
    times_clt = times.to_datetime(timezone=clt)

    crossings = []
    for i in range(1, len(sun_alt)):
        prev_alt = sun_alt[i - 1]
        curr_alt = sun_alt[i]
        if rising and prev_alt < altitude_deg <= curr_alt:
            crossings.append(times_clt[i])
        elif not rising and prev_alt > altitude_deg >= curr_alt:
            crossings.append(times_clt[i])

    return crossings


### EFD Query

In [ ]:
# Your datetime object (must be timezone-aware and in UTC)
year = int(str(dayobs)[:4])
month = int(str(dayobs)[4:6])
day = int(str(dayobs)[6:])
start_dt = datetime(year, month, day, 22, 0, 0, tzinfo=timezone.utc)

# Convert to astropy Time
start = Time(start_dt)

# Do the same for end if needed
end = start + TimeDelta(86400, format='sec')
try:
    efd_results = await asyncio.create_task(query_efd(start, end))
except KeyError:
    efd_results = None
    print("EFD results are empty")

### ConsDB query

In [ ]:
os.environ["no_proxy"] += ",.consdb"
consdb_url = 'http://consdb-pq.consdb:8080/consdb'
cdb_client = ConsDbClient(consdb_url)

In [ ]:
query = f"""
    SELECT
    e.airmass AS airmass,
    e.dimm_seeing AS dimm,
    e.altitude AS elevation,
    e.azimuth AS azimuth,
    e.exposure_id AS visit_id,
    e.physical_filter as band,
    e.day_obs AS day_obs,
    e.exp_midpt AS time,
    e.dimm_seeing AS seeing,
    e.science_program AS science_program,
    ccdvisit1_quicklook.psf_sigma,
    ccdvisit1.detector as detector,
    q.psf_sigma_median AS psf_fwhm,
    q.psf_sigma_min AS psf_fwhm_min,
    q.psf_sigma_max AS psf_fwhm_max,
    q.aos_fwhm AS aos_fwhm,
    q.donut_blur_fwhm as donut_blur_fwhm,
    e.obs_end,
    e.obs_start,
    e.seq_num AS seq
    FROM
    cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
    cdb_lsstcam.ccdvisit1 AS ccdvisit1,
    cdb_lsstcam.visit1 AS visit1,
    cdb_lsstcam.visit1_quicklook AS q,
    cdb_lsstcam.exposure AS e
    WHERE
    ccdvisit1.detector IN (191, 192, 195, 196, 199, 200, 203, 204)
    AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
    AND ccdvisit1.visit_id = visit1.visit_id
    AND ccdvisit1.visit_id = q.visit_id
    AND ccdvisit1.visit_id = e.exposure_id
    AND (e.img_type = 'science' or e.img_type = 'acq' or e.img_type = 'cwfs')
    AND e.exp_midpt > '{start.isot}'
    AND e.exp_midpt < '{end.isot}'
    AND e.airmass > 0
    AND e.band != 'none'
"""
cdb_table = cdb_client.query(query).to_pandas()

# Convert PSF sigma to FWHM
sig2fwhm = 2 * np.sqrt(2 * np.log(2))
pixel_scale = 0.2  # arcsec / pixel
cdb_table["psf_fwhm"] = cdb_table["psf_fwhm"] * sig2fwhm * pixel_scale

cdb_table["fwhm_zenith_500nm"] = [
    fwhm
    * getAirmassSeeingCorrection(airmass)
    * getBandpassSeeingCorrection(band)
    for fwhm, band, airmass in zip(
        cdb_table["psf_fwhm"], cdb_table["band"], cdb_table["airmass"]
    )
]

In [ ]:
# Group consdb data similarly to efd data
cdb_time_table = convert_dataframe_to_timezone(cdb_table, timestamp_column='obs_start')
cdb_sub_table = cdb_time_table[['dimm', 'aos_fwhm', 'fwhm_zenith_500nm', 'donut_blur_fwhm', 'band', 'science_program', 'seq']]
if len(cdb_sub_table)>0:
    cdb_sub_table = cdb_sub_table.groupby('obs_start').agg({'dimm': 'mean', 'aos_fwhm': 'mean', 'fwhm_zenith_500nm': 'mean', 'donut_blur_fwhm': 'mean', 'band': 'first', 'science_program': 'first', 'seq': 'first'})
else:
    print(f'No AOS data in consDB for  {dayobs}')
    

## Define plotting functions

In [ ]:
def plot_temperature_mtm1m3ts(ax, temperature_mtm1m3ts):
    mean_temp = temperature_mtm1m3ts.mean(axis=1)
    min_temp = temperature_mtm1m3ts.min(axis=1)
    max_temp = temperature_mtm1m3ts.max(axis=1)

    lower_pct = temperature_mtm1m3ts.quantile(0.1587, axis=1)
    upper_pct = temperature_mtm1m3ts.quantile(0.8413, axis=1)

    # Min–max band (light teal)
    ax.fill_between(
        temperature_mtm1m3ts.index,
        min_temp,
        max_temp,
        color="#c7ecee",  # light teal
        alpha=0.6,
        zorder=1,
    )

    # ±1 std band (medium teal)
    ax.fill_between(
        temperature_mtm1m3ts.index,
        lower_pct,
        upper_pct,
        color="#76d7c4",  # medium teal
        alpha=0.8,
        zorder=2,
    )

    # Mean line (deep teal)
    ax.plot(
        temperature_mtm1m3ts.index,
        mean_temp,
        color="#117864",  # deep teal
        label="M1M3 Temperature",
        zorder=3,
    )

band_colors = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}

def annotate_bands(data: pd.DataFrame, ax: plt.Axes) -> None:
    # Get the axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # Use actual datetime index instead of integer positions
    timestamps = data.index.values
    
    if len(timestamps) == 0:
        return
        
    # Create time intervals for each band
    for i, (timestamp, band) in enumerate(zip(timestamps, data['band'].values)):
        # Calculate width for each band segment
        if i < len(timestamps) - 1:
            next_timestamp = timestamps[i + 1]
            width = (next_timestamp - timestamp)
        else:
            # For last point, use same width as previous
            if i > 0:
                prev_timestamp = timestamps[i - 1]
                width = (timestamp - prev_timestamp)
        
        # Plot band bar
        ax.barh(
            ylim[0] + (ylim[1] - ylim[0]) * 0.02,  # slightly above bottom
            width,
            left=timestamp,
            height=(ylim[1] - ylim[0]) * 0.03,  # 3% of plot height
            color=band_colors[band[0]],
            alpha=0.8
        )

    # Restore the axis limits
    ax.set_ylim(ylim)
    ax.set_xlim(xlim)

    # Add unique bands to legend
    unique_bands = data['band'].str[0].unique()  # Get first character of each band
    
    for band_char in sorted(unique_bands):
        ax.scatter([], [], 
                  c=band_colors[band_char], 
                  s=100, 
                  marker='s', 
                  label=f'Band {band_char}')
    
def draw_plot(
    start_date: Time,
    end_date: Time,
    *,
    temperature_outdoor: pd.DataFrame,
    temperature_indoor: pd.DataFrame,
    temperature_mtm1m3ts: pd.DataFrame,
    temperature_mtm2: pd.DataFrame,
    temperature_tma_truss: pd.DataFrame,
    temperature_topend: pd.DataFrame,
    mtm1m3ts_setpoints: pd.DataFrame,
    topend_setpoints: pd.DataFrame,
    hvac_setpoints: pd.DataFrame,
    shutter: pd.DataFrame,
):

    # Create subplots with shared x-axis
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12), sharex=True)
    if not shutter.empty:
        closed_intervals = get_shutter_closed_intervals(shutter)

        # Plot background shading for closed periods on both subplots
        for start, end in closed_intervals:
            ax1.axvspan(start, end, facecolor="lightgray", alpha=0.3, zorder=0, label="_dome_closed")
            ax2.axvspan(start, end, facecolor="lightgray", alpha=0.3, zorder=0, label="_dome_closed")
    
        if closed_intervals:
            ax1.axvspan(
                closed_intervals[0][0],
                closed_intervals[0][0],  # dummy zero-width span just for legend
                facecolor="lightgray",
                alpha=0.3,
                label="Dome Closed",
            )

    # --- FIRST SUBPLOT: Temperature data ---
    # Line plots: outdoor and indoor temperatures
    ax1.plot(
        temperature_outdoor.index,
        temperature_outdoor["temperatureItem0"],
        label="Outdoor Temp (Index 301)",
    )
    ax1.plot(
        temperature_indoor.index,
        temperature_indoor["temperatureItem0"],
        label="Indoor Temp (Index 112)",
    )

    plot_temperature_mtm1m3ts(ax1, temperature_mtm1m3ts)

    ax1.plot(
        temperature_mtm2.index,
        temperature_mtm2["temperature"],
        label="M2 Temp (exc. Ring5)",
        color="magenta",
    )

    ax1.plot(
        temperature_mtm2.index,
        temperature_mtm2["ring5"],
        label="M2 Ambient Temp (Ring5)",
        color="crimson",
    )   

    ax1.plot(
        temperature_topend.index,
        temperature_topend["ambientTemperatureSensor0502"],
        label="Top End Temperature (Index 0502)",
        color="sienna",
    )

    if not temperature_tma_truss.empty:
        ax1.plot(
            temperature_tma_truss.index,
            temperature_tma_truss['tma_truss_+x_+y'],
            label='TMA Truss Temperature (+x,+y)',
            color='turquoise'
        )
    
        ax1.plot(
            temperature_tma_truss.index,
            temperature_tma_truss['tma_truss_-x_-y'],
            label='TMA Truss Temperature (-x,-y)',
            color='navy'
        )
    
    # Top End Setpoints
    if not topend_setpoints.empty:
        topend = topend_setpoints.copy()
        topend["adjusted"] = (
            topend["topEndChillerSetpoint"]
        )

        for identity in topend["private_identity"].unique():
            mask = topend["private_identity"] == identity
            ax1.scatter(
                topend.index[mask],
                topend["adjusted"][mask],
                label="Top End Setpoint" if identity == "EAS" else None,
                color="tab:red" if identity == "EAS" else "gray",
                marker="x",
                zorder=4,
            )

    # HVAC Setpoints (if present)
    if not hvac_setpoints.empty:
        hvac = hvac_setpoints.copy()
        hvac["adjusted"] = hvac["workingSetpoint"]
        ax1.plot(hvac.index, hvac["adjusted"], linestyle=":", color="black", alpha=0.4)

        for identity in hvac["private_identity"].unique():
            mask = hvac["private_identity"] == identity
            ax1.scatter(
                hvac.index[mask],
                hvac["adjusted"][mask],
                label="HVAC Setpoint" if identity == "EAS" else None,
                color="tab:purple" if identity == "EAS" else "gray",
                marker="s",
            )

    # MTM1M3TS Setpoints
    if not mtm1m3ts_setpoints.empty:
        mtm1 = mtm1m3ts_setpoints.copy()
        mtm1["adjusted"] = mtm1["heatersSetpoint"] # no adjustment. Heaters set point should track ~1 deg below ambient
        ax1.plot(mtm1.index, mtm1["adjusted"], linestyle=":", color="black", alpha=0.4, zorder=2)

        for identity in mtm1["private_identity"].unique():
            mask = mtm1["private_identity"] == identity
            ax1.scatter(
                mtm1.index[mask],
                mtm1["adjusted"][mask],
                label="M1M3 Setpoint" if identity == "EAS" else None,
                color="tab:green" if identity == "EAS" else "gray",
                marker="o",
                s=100,  # larger size
                zorder=5,  # ensure it's on top
                alpha=0.5,
            )

    twilight_ends = find_sun_altitude_crossings(start_date, end_date, altitude_deg=-18, rising=False)
    sunrises = find_sun_altitude_crossings(start_date, end_date, altitude_deg=0, rising=True)

    if twilight_ends:
        ax1.axvline(twilight_ends[0], color="black", linestyle="--", linewidth=2.5, label="Twilight End")
        ax2.axvline(twilight_ends[0], color="black", linestyle="--", linewidth=2.5, label="Twilight End")
    if sunrises:
        ax1.axvline(sunrises[0], color="orange", linestyle="--", linewidth=2.5, label="Sunrise")
        ax2.axvline(sunrises[0], color="orange", linestyle="--", linewidth=2.5, label="Sunrise")

    # First subplot formatting
    ax1.set_ylabel("Temperature (°C)")
    ax1.legend(frameon=True, framealpha=1.0, facecolor='white', ncol=2, fontsize=9)

    # --- SECOND SUBPLOT: Image Quality data ---
    ax2.plot(cdb_sub_table.index, cdb_sub_table['fwhm_zenith_500nm'],  label='fwhm_zenith_500nm')
    ax2.plot(cdb_sub_table.index, cdb_sub_table['dimm'], label='dimm')
    ax2.plot(cdb_sub_table.index, cdb_sub_table['aos_fwhm'], label='aos_fwhm')
    ax2.plot(cdb_sub_table.index, cdb_sub_table['donut_blur_fwhm'],label='Donut Blur FWHM')

    ax2.set_ylabel('FWHM (arcsec)')
    ax2.set_title('Image Quality Measurements')

    # Add shading for survey data
    survey_intervals = get_survey_intervals(cdb_sub_table)

    # Plot background shading for closed periods on both subplots
    for start, end in survey_intervals:
        ax2.axvspan(start, end, facecolor="pink", alpha=0.3, zorder=0, label="_survey_observations")

    if survey_intervals:
        ax2.axvspan(
            survey_intervals[0][0],
            survey_intervals[0][0],  # dummy zero-width span just for legend
            facecolor="pink",
            alpha=0.3,
            label="Survey Observations",
        )

    annotate_bands(cdb_sub_table, ax2)
    
    ax2.legend(frameon=True, framealpha=1.0, facecolor='white')

    # Format datetime x-axis ticks (only on bottom subplot)
    locator = mdates.AutoDateLocator()
    formatter = mdates.ConciseDateFormatter(locator)
    ax2.xaxis.set_major_locator(locator)
    ax2.xaxis.set_major_formatter(formatter)
    ax2.set_xlabel("Time (UTC)")

    # Add start date to title (on top subplot)
    start_dt_str = start_date.to_datetime().strftime("%Y-%m-%d")
    ax1.set_title(f"EAS Summary – {start_dt_str}")
    
    # fig.tight_layout()
    return fig, (ax1, ax2)


## Draw Plot

In [ ]:
if efd_results:
    fig, ax = draw_plot(start, end, **efd_results)
else:
    print("No data to plot")